# Topic 22a - Asking good questions

The main lesson showed one vague-to-specific rewrite. This notebook practices the skill directly: turning curiosity into questions a column can actually answer, and recognising the four question shapes that map onto the chart types from Topic 21.

Work through it after reading section 2 of the [main lesson](topic22_exploratory_data_analysis.ipynb).

In [1]:
# Run this cell as it is.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_data_dir():
    """Locate the lesson's data folder from the repository root, the lesson
    folder, or the assignments folder, so this notebook runs from anywhere."""
    candidates = [
        Path.cwd() / "data",
        Path.cwd() / "exploratory_data_analysis" / "data",
        Path.cwd().parent / "data",
    ]
    for candidate in candidates:
        if (candidate / "air_quality_readings.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find air_quality_readings.csv. Looked in: " +
        ", ".join(str(c) for c in candidates)
    )


DATA_DIR = find_data_dir()

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 130)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#e1e0d9"
plt.rcParams["grid.linewidth"] = 0.8
plt.rcParams["axes.edgecolor"] = "#c3c2b7"
plt.rcParams["axes.axisbelow"] = True

# The course palette (Topic 21) - fixed hue order, never cycled, so a category
# keeps the same colour in every chart it appears in.
PALETTE = {
    "blue": "#2a78d6", "aqua": "#1baf7a", "yellow": "#eda100", "green": "#008300",
    "violet": "#4a3aa7", "red": "#e34948", "magenta": "#e87ba4", "orange": "#eb6834",
}
MUTED = "#c3c2b7"
INK = "#0b0b0b"
STATION_TYPE_COLOR = {
    "Roadside": PALETTE["red"], "Industrial": PALETTE["orange"],
    "Residential": PALETTE["aqua"], "Background": PALETTE["blue"],
}
STATION_TYPE_ORDER = ["Roadside", "Industrial", "Residential", "Background"]

WHO_ANNUAL_GUIDELINE = 5.0   # ug/m3, PM2.5 annual mean
WHO_24H_GUIDELINE = 15.0     # ug/m3, PM2.5 24-hour mean

print("pandas version:   ", pd.__version__)
print("data folder:      ", DATA_DIR)
print("files:            ", sorted(p.name for p in DATA_DIR.glob("*.csv")))

pandas version:    3.0.6
data folder:       /home/johnny/Projects/itfy-da/exploratory_data_analysis/data
files:             ['air_quality_readings.csv', 'air_quality_readings_assignment.csv', 'air_quality_stations.csv', 'air_quality_stations_assignment.csv', 'customers.csv', 'monthly_targets_wide.csv', 'orders.csv', 'products.csv', 'regions.csv', 'returns.csv']


In [2]:
stations = pd.read_csv(DATA_DIR / "air_quality_stations.csv")
readings = pd.read_csv(DATA_DIR / "air_quality_readings.csv", parse_dates=["date"])

aq = readings.merge(stations, on="station_id", how="left", validate="m:1")
aq["month"] = aq["date"].dt.month
aq["is_weekend"] = aq["date"].dt.dayofweek.isin([5, 6])
aq["is_harmattan"] = aq["month"].isin([12, 1, 2])

print(f"Air-quality table: {aq.shape[0]} rows x {aq.shape[1]} columns")
print(f"Stations: {aq['station_id'].nunique()}, dates: {aq['date'].min().date()} to {aq['date'].max().date()}")
print(f"Missing pm2_5 readings: {aq['pm2_5'].isna().sum()}")

Air-quality table: 3285 rows x 14 columns
Stations: 9, dates: 2023-01-01 to 2023-12-31
Missing pm2_5 readings: 7


---

## The four question shapes

Every answerable EDA question is one of four shapes, and each shape points straight at a Topic 21 chart type:

| Question shape | Example | Chart |
|---|---|---|
| Comparison | "Which station type has the worst PM2.5?" | Bar or box plot |
| Trend | "Did PM2.5 change over the year?" | Line |
| Relationship | "Does wind speed affect PM2.5?" | Scatter |
| Distribution | "What does a typical day's PM2.5 look like, and how much does it vary?" | Histogram or box plot |

Knowing the shape before you write any code tells you which chart to reach for - you are applying Topic 21's chart-selection rule one step earlier than before, to a question you just wrote yourself instead of one someone handed you.

### Try It 1 - name the shape

For each question below, name its shape (comparison / trend / relationship / distribution) and the chart you would build:

1. "Is NO2 higher on weekdays than weekends?"
2. "How does PM2.5 vary month to month?"
3. "Is there a relationship between temperature and humidity?"
4. "What is the spread of wind speeds across the whole year?"

Write your four answers as a comment, then pick **one** of the four and actually answer it with code.

In [3]:
# YOUR CODE HERE

---

## Rewriting vague curiosity

A specific question names a **variable**, what it is being **compared against**, and is **answerable from columns you have**. Practice the rewrite on three more vague prompts.

In [4]:
vague_prompts = [
    "Is Accra's air quality getting better or worse?",
    "Are industrial areas worse than other places?",
    "Does weather affect pollution?",
]
for p in vague_prompts:
    print("Vague:   ", p)
print()
print("None of these name a time window, a specific comparison, or a specific pollutant.")
print("A rewrite of the first one: 'Was mean PM2.5 in the second half of 2023 higher or")
print("lower than the first half?' - specific, comparative, and answerable from `aq` alone.")

Vague:    Is Accra's air quality getting better or worse?
Vague:    Are industrial areas worse than other places?
Vague:    Does weather affect pollution?

None of these name a time window, a specific comparison, or a specific pollutant.
A rewrite of the first one: 'Was mean PM2.5 in the second half of 2023 higher or
lower than the first half?' - specific, comparative, and answerable from `aq` alone.


### Try It 2 - your own rewrites

Rewrite the second and third vague prompts above into specific, answerable questions, following the same pattern. Then answer both with code.

In [5]:
# YOUR CODE HERE

---

## When a question turns out to be two questions

"Is NO2 higher on weekdays?" looks like one comparison question. It is actually hiding a second one: *is that true everywhere, or only at some station types?* The second version is more useful, because the answer changes what you would recommend.

In [6]:
overall = aq.groupby("is_weekend")["no2"].mean()
print("Overall weekday vs weekend NO2:")
print(overall.round(1).to_string())
print()
by_type = aq.groupby(["station_type", "is_weekend"])["no2"].mean().unstack()
by_type.columns = ["Weekday", "Weekend"]
print("Broken down by station type:")
print(by_type.round(1).to_string())

Overall weekday vs weekend NO2:
is_weekend
False    26.3
True     19.8

Broken down by station type:
              Weekday  Weekend
station_type                  
Background        9.1      6.8
Industrial       30.0     22.5
Residential      18.0     13.7
Roadside         37.9     28.6


The weekday effect holds at every station type here - the gap is not hiding a more interesting split. That is itself worth noting in a report: "this is a citywide pattern, not a one-neighbourhood effect" is a stronger, more useful claim than the first version.

---
## Where this fits

- [Back to the main lesson](topic22_exploratory_data_analysis.ipynb)
- Previous: [Main lesson](topic22_exploratory_data_analysis.ipynb)
- Next: [22b - Univariate exploration](topic22b_univariate_exploration.ipynb)